# Fetch News

Bulk-imports Benzinga articles from the Massive API into the article store
(`data/news/`).  Articles are fetched without a ticker filter and stored as
monthly parquet files.  The cell is safe to re-run — articles already on disk
(matched by `benzinga_id`) are skipped.

**Prerequisites:**
- `.env` with `MASSIVE_API_KEY`

In [ ]:
from __future__ import annotations

import logging

import src  # triggers load_dotenv()
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("fetch_news")

## Config

In [ ]:
from datetime import date

# Inclusive date range to fetch.  Fetching is done month by month so this
# can span multiple months without loading everything into memory at once.
START_DATE = date(2018, 1, 1)
END_DATE   = date(2019, 12, 31)

# API page size — 50 000 is the practical ceiling for a single month.
PAGE_LIMIT = 50_000

## Fetch

Iterates month by month over the configured range.  Each iteration fetches all
articles published in that month and writes them to `data/news/<YYYY-MM>.parquet`.

In [ ]:
import calendar

from src.providers.massive import MassiveProvider
from src.repositories.articles import ArticleRepository

provider     = MassiveProvider()
article_repo = ArticleRepository()

year, month = START_DATE.year, START_DATE.month
end_year, end_month = END_DATE.year, END_DATE.month

while (year, month) <= (end_year, end_month):
    month_start = date(year, month, 1)
    month_end   = date(year, month, calendar.monthrange(year, month)[1])

    # Clamp to the configured window
    fetch_start = max(month_start, START_DATE)
    fetch_end   = min(month_end, END_DATE)

    logger.info("Fetching %s-%02d  (%s → %s)", year, month, fetch_start, fetch_end)

    articles = provider.fetch_all_news(fetch_start, fetch_end, limit=PAGE_LIMIT)
    article_repo.bulk_store(articles)

    logger.info("%d-%02d: stored %d articles", year, month, len(articles))

    month += 1
    if month > 12:
        month = 1
        year += 1

print("Done.")

In [ ]:
df = article_repo.read_month(2026, 1)